In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/tamirka/notebook8c09694358/__results__.html
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_classification.csv
/kaggle/input/notebooks/tamirka/notebook8c09694358/__notebook__.ipynb
/kaggle/input/notebooks/tamirka/notebook8c09694358/__output__.json
/kaggle/input/notebooks/tamirka/notebook8c09694358/custom.css
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_buy_sell_lifecycle.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_realized_pnl_trades.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_hold_stats.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/wallet_realized_pnl_stats.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/events_clean.parquet
/kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/price_events.pa

In [2]:
from pathlib import Path
import os

BASE = Path("/kaggle/input/notebooks/tamirka/notebook8c09694358")

print("BASE exists:", BASE.exists())

for root, dirs, files in os.walk(BASE):
    level = root.replace(str(BASE), "").count(os.sep)
    if level <= 2:
        print("\n", root)
        for d in dirs[:20]:
            print("  DIR:", d)
        for f in files[:20]:
            print("  FILE:", f)

BASE exists: True

 /kaggle/input/notebooks/tamirka/notebook8c09694358
  DIR: wallet_type_detector_cache_v2
  DIR: wallet_type_detector_exports_v2
  FILE: __results__.html
  FILE: wallet_type_classification.csv
  FILE: __notebook__.ipynb
  FILE: __output__.json
  FILE: custom.css

 /kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2
  DIR: lifecycle_checkpoints_fast
  FILE: wallet_buy_sell_lifecycle.parquet
  FILE: wallet_realized_pnl_trades.parquet
  FILE: wallet_hold_stats.parquet
  FILE: wallet_realized_pnl_stats.parquet
  FILE: events_clean.parquet
  FILE: price_events.parquet
  FILE: fb_first_buys.parquet

 /kaggle/input/notebooks/tamirka/notebook8c09694358/wallet_type_detector_cache_v2/lifecycle_checkpoints_fast
  FILE: lifecycle_part_00008.parquet
  FILE: lifecycle_part_00011.parquet
  FILE: lifecycle_part_00019.parquet
  FILE: lifecycle_part_00015.parquet
  FILE: lifecycle_part_00022.parquet
  FILE: lifecycle_part_00003.parquet
  FILE: lifecycle_par

In [3]:
from pathlib import Path
import shutil

BASE = Path("/kaggle/input/notebooks/tamirka/notebook8c09694358")

INPUT_CACHE = BASE / "wallet_type_detector_cache_v2"
WORK_CACHE = Path("/kaggle/working/wallet_type_detector_cache_v2")

INPUT_EXPORTS = BASE / "wallet_type_detector_exports_v2"
WORK_EXPORTS = Path("/kaggle/working/exports_token_events_copy")

if WORK_CACHE.exists():
    shutil.rmtree(WORK_CACHE)
shutil.copytree(INPUT_CACHE, WORK_CACHE)

if WORK_EXPORTS.exists():
    shutil.rmtree(WORK_EXPORTS)
shutil.copytree(INPUT_EXPORTS, WORK_EXPORTS)

print("cache restored:", WORK_CACHE)
print("exports restored:", WORK_EXPORTS)
print("cache files:", len(list(WORK_CACHE.rglob("*"))))
print("export files:", len(list(WORK_EXPORTS.rglob("*"))))

cache restored: /kaggle/working/wallet_type_detector_cache_v2
exports restored: /kaggle/working/exports_token_events_copy
cache files: 34
export files: 9


In [4]:
# ============================================================
# 02 — REBUILD CLEAN FIRST-SIGNAL DATASET
# Creates fresh_signals_clean.parquet
# One first BUY signal per token, age 5–120s
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

CACHE_DIR = Path("/kaggle/working/wallet_type_detector_cache_v2")
EXPORT_DIR = Path("/kaggle/working/wallet_type_detector_exports_v2")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Use already loaded events if available, otherwise load
try:
    events
except NameError:
    events = pd.read_parquet(CACHE_DIR / "events_clean.parquet")

events = events.copy()
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
events["launchTime"] = pd.to_datetime(events["launchTime"], errors="coerce")

SIGNAL_AGE_MIN = 5
SIGNAL_AGE_MAX = 120
FUTURE_WINDOW = 180
PRE_WINDOWS = [5, 10, 30, 60]

token_groups = {
    token: g.sort_values("timestamp").reset_index(drop=True)
    for token, g in events.groupby("tokenAddr", sort=False)
}

print("tokens:", len(token_groups))

def get_window(g, t0, sec_back):
    start = t0 - pd.Timedelta(seconds=sec_back)
    return g[(g["timestamp"] >= start) & (g["timestamp"] <= t0)]

def get_future(g, t0, sec_forward):
    end = t0 + pd.Timedelta(seconds=sec_forward)
    return g[(g["timestamp"] >= t0) & (g["timestamp"] <= end)]

def pre_features_clean(g, t0, entry_price):
    feats = {}

    for sec in PRE_WINDOWS:
        w = get_window(g, t0, sec)
        buys = w[w["eventType"] == "BUY"]
        sells = w[w["eventType"] == "SELL"]

        bvol = buys["amountSol"].sum()
        svol = sells["amountSol"].sum()

        feats[f"buyVol{sec}s"] = bvol
        feats[f"sellVol{sec}s"] = svol
        feats[f"ratio{sec}s"] = bvol / svol if svol > 0 else np.inf

        feats[f"buyCount{sec}s"] = len(buys)
        feats[f"sellCount{sec}s"] = len(sells)
        feats[f"uniqueBuyers{sec}s"] = buys["wallet"].nunique()
        feats[f"uniqueSellers{sec}s"] = sells["wallet"].nunique()

        if len(w) > 0 and entry_price > 0:
            first_price = w.iloc[0]["price"] if w.iloc[0]["price"] > 0 else entry_price
            max_price = w["price"].max()
            min_price = w["price"].min()

            feats[f"delta{sec}sPct"] = ((entry_price - first_price) / first_price) * 100 if first_price > 0 else 0
            feats[f"maxPrice{sec}s"] = max_price
            feats[f"minPrice{sec}s"] = min_price
            feats[f"drawdown{sec}sPct"] = ((entry_price - max_price) / max_price) * 100 if max_price > 0 else 0
            feats[f"recoveryFromMin{sec}sPct"] = ((entry_price - min_price) / min_price) * 100 if min_price > 0 else 0
        else:
            feats[f"delta{sec}sPct"] = 0
            feats[f"maxPrice{sec}s"] = entry_price
            feats[f"minPrice{sec}s"] = entry_price
            feats[f"drawdown{sec}sPct"] = 0
            feats[f"recoveryFromMin{sec}sPct"] = 0

        if len(buys) > 0 and bvol > 0:
            top = buys.groupby("wallet")["amountSol"].sum().sort_values(ascending=False)
            feats[f"top1BuyerShare{sec}s"] = top.iloc[0] / bvol * 100
            feats[f"top3BuyerShare{sec}s"] = top.head(3).sum() / bvol * 100
            feats[f"top5BuyerShare{sec}s"] = top.head(5).sum() / bvol * 100
            feats[f"largestBuyerShare{sec}s"] = top.iloc[0] / bvol * 100
        else:
            feats[f"top1BuyerShare{sec}s"] = 0
            feats[f"top3BuyerShare{sec}s"] = 0
            feats[f"top5BuyerShare{sec}s"] = 0
            feats[f"largestBuyerShare{sec}s"] = 0

    return feats

def future_outcome_clean(g, t0, entry_price):
    f = get_future(g, t0, FUTURE_WINDOW)
    out = {}

    if f.empty or entry_price <= 0:
        out["maxProfit180s"] = 0
        out["maxDrawdown180s"] = 0
        out["finalPct180s"] = 0
        for tp in [10, 20, 30, 50, 70, 100, 120, 150]:
            out[f"reach{tp}"] = False
        out["dead"] = True
        out["stopFirst20"] = False
        out["stopFirst30"] = False
        return out

    max_p = f["price"].max()
    min_p = f["price"].min()

    out["maxProfit180s"] = ((max_p - entry_price) / entry_price) * 100
    out["maxDrawdown180s"] = ((min_p - entry_price) / entry_price) * 100
    out["finalPct180s"] = ((f.iloc[-1]["price"] - entry_price) / entry_price) * 100

    for tp in [10, 20, 30, 50, 70, 100, 120, 150]:
        out[f"reach{tp}"] = out["maxProfit180s"] >= tp

    out["dead"] = out["maxProfit180s"] < 5

    stop20 = False
    stop30 = False

    for _, r in f.sort_values("timestamp").iterrows():
        p = ((r["price"] - entry_price) / entry_price) * 100

        if p <= -20:
            stop20 = True
        if p <= -30:
            stop30 = True

        if p >= 20:
            break

    out["stopFirst20"] = stop20
    out["stopFirst30"] = stop30

    return out

rows = []
tokens = list(token_groups.keys())

for idx, token in enumerate(tokens):
    if idx % 1000 == 0:
        print(f"processed {idx}/{len(tokens)}")

    g = token_groups[token]

    early = g[
        (g["ageSec"] >= SIGNAL_AGE_MIN) &
        (g["ageSec"] <= SIGNAL_AGE_MAX) &
        (g["eventType"] == "BUY")
    ]

    if early.empty:
        continue

    # first BUY signal only per token
    ev = early.iloc[0]
    t0 = ev["timestamp"]
    entry_price = ev["price"]

    if pd.isna(t0) or pd.isna(entry_price) or entry_price <= 0:
        continue

    feats = pre_features_clean(g, t0, entry_price)
    outcome = future_outcome_clean(g, t0, entry_price)

    row = {
        "tokenAddr": token,
        "entryTime": t0,
        "entryAgeSec": ev["ageSec"],
        "entryPrice": entry_price,
        "entryBuySol": ev["amountSol"],
        "entryWallet": ev["wallet"],
        "txSignature": ev["txSignature"],
        "slot": ev["slot"],
        "launchTime": ev["launchTime"],
    }

    row.update(feats)
    row.update(outcome)
    rows.append(row)

signals_clean = pd.DataFrame(rows)

print("signals_clean shape:", signals_clean.shape)
print("reach20:", signals_clean["reach20"].mean() * 100)
print("reach50:", signals_clean["reach50"].mean() * 100)
print("reach100:", signals_clean["reach100"].mean() * 100)
print("reach120:", signals_clean["reach120"].mean() * 100)
print("dead:", signals_clean["dead"].mean() * 100)
print("stopFirst20:", signals_clean["stopFirst20"].mean() * 100)
print("stopFirst30:", signals_clean["stopFirst30"].mean() * 100)

signals_clean.to_parquet(CACHE_DIR / "fresh_signals_clean.parquet", index=False)
print("saved:", CACHE_DIR / "fresh_signals_clean.parquet")

tokens: 157651
processed 0/157651
processed 1000/157651
processed 2000/157651
processed 3000/157651
processed 4000/157651
processed 5000/157651
processed 6000/157651
processed 7000/157651
processed 8000/157651
processed 9000/157651
processed 10000/157651
processed 11000/157651
processed 12000/157651
processed 13000/157651
processed 14000/157651
processed 15000/157651
processed 16000/157651
processed 17000/157651
processed 18000/157651
processed 19000/157651
processed 20000/157651
processed 21000/157651
processed 22000/157651
processed 23000/157651
processed 24000/157651
processed 25000/157651
processed 26000/157651
processed 27000/157651
processed 28000/157651
processed 29000/157651
processed 30000/157651
processed 31000/157651
processed 32000/157651
processed 33000/157651
processed 34000/157651
processed 35000/157651
processed 36000/157651
processed 37000/157651
processed 38000/157651
processed 39000/157651
processed 40000/157651
processed 41000/157651
processed 42000/157651
processed

In [5]:
# ============================================================
# 08 — TEST TRUE_BEST REBUILD FROM fb_first_buys
# Goal: check if old E strategy came from wallet first-buy rows
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

CACHE_DIR = Path("/kaggle/working/wallet_type_detector_cache_v2")

fb = pd.read_parquet(CACHE_DIR / "fb_first_buys.parquet").copy()

try:
    events
except NameError:
    events = pd.read_parquet(CACHE_DIR / "events_clean.parquet")

fb["walletBuyTime"] = pd.to_datetime(fb["walletBuyTime"], errors="coerce")
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")

print("fb shape:", fb.shape)
print("fb columns:", list(fb.columns))

print("\nFB AGE CHECK")
print(fb["walletBuyAgeSec"].describe())

print("\nFB MAX PROFIT CHECK")
print(fb["maxProfit120s"].describe())

# Basic old-style first-signal entry zone
base = fb[
    (fb["walletBuyAgeSec"] >= 5) &
    (fb["walletBuyAgeSec"] <= 10) &
    (fb["walletBuyPrice"] > 0) &
    (fb["walletBuySol"] > 0)
].copy()

print("\nbase fb age 5-10:", base.shape)

print("reach20:", (base["maxProfit120s"] >= 20).mean() * 100)
print("reach50:", (base["maxProfit120s"] >= 50).mean() * 100)
print("reach100:", (base["maxProfit120s"] >= 100).mean() * 100)
print("reach120:", (base["maxProfit120s"] >= 120).mean() * 100)
print("dead:", (base["maxProfit120s"] < 5).mean() * 100)

# Build event groups for path exit simulation
token_groups = {
    token: g.sort_values("timestamp").reset_index(drop=True)
    for token, g in events.groupby("tokenAddr", sort=False)
}

def simulate_exit_path(token, entry_time, entry_price, hard_stop=-30, take_profit=120, max_hold_sec=180):
    g = token_groups.get(token)

    if g is None or len(g) == 0 or entry_price <= 0:
        return {"exitPct": 0, "reason": "no_data", "holdSec": 0}

    entry_time = pd.to_datetime(entry_time)
    end_time = entry_time + pd.Timedelta(seconds=max_hold_sec)

    f = g[
        (g["timestamp"] >= entry_time) &
        (g["timestamp"] <= end_time)
    ].sort_values("timestamp")

    if f.empty:
        return {"exitPct": 0, "reason": "no_future", "holdSec": 0}

    for _, r in f.iterrows():
        price = r["price"]

        if pd.isna(price) or price <= 0:
            continue

        pct = ((price - entry_price) / entry_price) * 100
        hold_sec = (r["timestamp"] - entry_time).total_seconds()

        if pct <= hard_stop:
            return {"exitPct": pct, "reason": "hard_stop", "holdSec": hold_sec}

        if pct >= take_profit:
            return {"exitPct": pct, "reason": "take_profit", "holdSec": hold_sec}

    final_price = f.iloc[-1]["price"]
    final_pct = ((final_price - entry_price) / entry_price) * 100
    hold_sec = (f.iloc[-1]["timestamp"] - entry_time).total_seconds()

    return {"exitPct": final_pct, "reason": "timeout", "holdSec": hold_sec}

# Test simple fb-based candidate buckets around old TRUE_BEST size
entry_models = {
    "FB_age5_10_all": base,
    "FB_age5_10_sol_le_0p10": base[base["walletBuySol"] <= 0.10],
    "FB_age5_10_sol_le_0p20": base[base["walletBuySol"] <= 0.20],
    "FB_age5_10_sol_0p01_0p20": base[(base["walletBuySol"] >= 0.01) & (base["walletBuySol"] <= 0.20)],
    "FB_age5_10_profit120_ge_20_RESEARCH_ONLY": base[base["maxProfit120s"] >= 20],
}

exit_configs = [
    {"name": "HS20_TP50_H2m",    "hard_stop": -20, "take_profit": 50,  "max_hold": 120},
    {"name": "HS25_TP70_H2m",    "hard_stop": -25, "take_profit": 70,  "max_hold": 120},
    {"name": "HS30_TP100_H3m",   "hard_stop": -30, "take_profit": 100, "max_hold": 180},
    {"name": "HS30_TP120_H3m",   "hard_stop": -30, "take_profit": 120, "max_hold": 180},
]

START_SOL = 0.03
FEE_SOL = 0.00005

summary_rows = []
all_rows = []

for model_name, entries in entry_models.items():
    entries = entries.copy().reset_index(drop=True)
    print("\nMODEL:", model_name, "entries:", len(entries))

    for cfg in exit_configs:
        print("testing:", model_name, cfg["name"])

        rows = []

        for _, r in entries.iterrows():
            sim = simulate_exit_path(
                token=r["tokenAddr"],
                entry_time=r["walletBuyTime"],
                entry_price=r["walletBuyPrice"],
                hard_stop=cfg["hard_stop"],
                take_profit=cfg["take_profit"],
                max_hold_sec=cfg["max_hold"],
            )

            pnl_sol = START_SOL * (sim["exitPct"] / 100) - FEE_SOL

            rows.append({
                "model": model_name,
                "exit": cfg["name"],
                "wallet": r["wallet"],
                "tokenAddr": r["tokenAddr"],
                "entryTime": r["walletBuyTime"],
                "entryAgeSec": r["walletBuyAgeSec"],
                "entryPrice": r["walletBuyPrice"],
                "entryBuySol": r["walletBuySol"],
                "txSignature": r["txSignature"],
                "exitPct": sim["exitPct"],
                "pnlSOL": pnl_sol,
                "reason": sim["reason"],
                "holdSec": sim["holdSec"],
                "maxProfit120s": r["maxProfit120s"],
            })

        out = pd.DataFrame(rows)
        all_rows.append(out)

        vc = out["reason"].value_counts(normalize=True) * 100
        winners = out[out["pnlSOL"] > 0]
        losers = out[out["pnlSOL"] <= 0]

        gross_profit = winners["pnlSOL"].sum()
        gross_loss = -losers["pnlSOL"].sum()
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

        summary_rows.append({
            "model": model_name,
            "exit": cfg["name"],
            "trades": len(out),
            "WR": (out["pnlSOL"] > 0).mean() * 100,
            "totalSOL": out["pnlSOL"].sum(),
            "SOLtrade": out["pnlSOL"].mean(),
            "profitFactor": profit_factor,
            "avgExitPct": out["exitPct"].mean(),
            "medianExitPct": out["exitPct"].median(),
            "tpRate": vc.get("take_profit", 0),
            "hardStopRate": vc.get("hard_stop", 0),
            "timeoutRate": vc.get("timeout", 0),
            "avgHoldSec": out["holdSec"].mean(),
            "medianHoldSec": out["holdSec"].median(),
        })

fb_exit_trades = pd.concat(all_rows, ignore_index=True)
fb_exit_summary = pd.DataFrame(summary_rows).sort_values("totalSOL", ascending=False).reset_index(drop=True)

display(fb_exit_summary)

fb_exit_trades.to_parquet(CACHE_DIR / "fb_based_exit_test_trades.parquet", index=False)
fb_exit_summary.to_csv(CACHE_DIR / "fb_based_exit_test_summary.csv", index=False)

print("saved trades:", CACHE_DIR / "fb_based_exit_test_trades.parquet", fb_exit_trades.shape)
print("saved summary:", CACHE_DIR / "fb_based_exit_test_summary.csv")

fb shape: (2425240, 13)
fb columns: ['wallet', 'tokenAddr', 'eventType', 'walletBuyTime', 'txSignature', 'walletBuySol', 'tokenAmount', 'walletBuyPrice', 'slot', 'metadata', 'launchTime', 'walletBuyAgeSec', 'maxProfit120s']

FB AGE CHECK
count    2.425240e+06
mean     6.410879e+03
std      3.072302e+04
min      0.000000e+00
25%      5.000000e+00
50%      3.700000e+01
75%      2.710000e+02
max      3.589330e+05
Name: walletBuyAgeSec, dtype: float64

FB MAX PROFIT CHECK
count    2.425240e+06
mean     4.635935e+01
std      1.456624e+02
min      0.000000e+00
25%      2.870778e+00
50%      1.702516e+01
75%      4.847013e+01
max      1.830775e+04
Name: maxProfit120s, dtype: float64

base fb age 5-10: (217803, 13)
reach20: 55.82567733226816
reach50: 33.898063846687144
reach100: 17.227494570781854
reach120: 13.540676666528928
dead: 23.71730416936406

MODEL: FB_age5_10_all entries: 217803
testing: FB_age5_10_all HS20_TP50_H2m
testing: FB_age5_10_all HS25_TP70_H2m
testing: FB_age5_10_all HS30_TP

,model,exit,trades,WR,totalSOL,SOLtrade,profitFactor,avgExitPct,medianExitPct,tpRate,hardStopRate,timeoutRate,avgHoldSec,medianHoldSec
0,FB_age5_10_profit120_ge_20_RESEARCH_ONLY,HS25_TP70_H2m,121590,41.697508,479.020585,0.003945,1.777352,13.316638,-25.131609,34.881158,51.482030,13.636812,29.945793,13.0
1,FB_age5_10_profit120_ge_20_RESEARCH_ONLY,HS30_TP100_H3m,121590,35.079365,452.903049,0.003739,1.593550,12.629430,-30.290222,27.084464,53.784851,19.130685,51.335932,24.0
2,FB_age5_10_profit120_ge_20_RESEARCH_ONLY,HS20_TP50_H2m,121590,45.538284,450.076406,0.003703,1.896215,12.511089,-20.047458,42.292129,50.513200,7.194671,19.030373,6.0
3,FB_age5_10_profit120_ge_20_RESEARCH_ONLY,HS30_TP120_H3m,121590,31.565918,404.031217,0.003341,1.503140,11.302617,-30.505167,21.676947,56.507114,21.815939,56.389325,28.0
4,FB_age5_10_sol_0p01_0p20,HS20_TP50_H2m,83291,27.678861,-22.489585,-0.000270,0.942506,-0.733601,-20.324430,23.258215,53.445150,23.296635,26.851232,8.0
5,FB_age5_10_sol_le_0p20,HS20_TP50_H2m,100576,27.485682,-33.347284,-0.000332,0.930083,-0.938807,-20.382233,23.133750,54.011892,22.854359,26.173282,8.0
6,FB_age5_10_sol_le_0p10,HS20_TP50_H2m,69587,27.575553,-37.073355,-0.000533,0.894875,-1.609745,-20.749950,23.534568,57.638639,18.826792,21.918979,6.0
7,FB_age5_10_sol_0p01_0p20,HS25_TP70_H2m,83291,25.401304,-42.064195,-0.000505,0.908353,-1.517767,-25.203151,18.939621,52.360999,28.699379,34.019690,13.0
8,FB_age5_10_sol_le_0p10,HS25_TP70_H2m,69587,25.447282,-52.147766,-0.000750,0.873543,-2.333058,-25.573778,19.486398,56.241827,24.271775,28.881716,11.0
9,FB_age5_10_sol_le_0p20,HS25_TP70_H2m,100576,25.183941,-58.485801,-0.000582,0.895530,-1.772814,-25.254688,18.852410,52.875437,28.272152,33.259396,13.0


saved trades: /kaggle/working/wallet_type_detector_cache_v2/fb_based_exit_test_trades.parquet (2371388, 14)
saved summary: /kaggle/working/wallet_type_detector_cache_v2/fb_based_exit_test_summary.csv
